## Подготовка

In [8]:
# Автоперезагрузка модулей из src при их изменении
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
# Базовые библиотеки
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

# Функции из src
from src.data import load_data, save_submission, build_preprocessor
from src.validation import cross_validate, log_result
from src.models import build_model
from src.features import add_features

import warnings
warnings.filterwarnings("ignore", message="Found unknown categories")

In [36]:
# Загрузка конфига и данных
config = OmegaConf.load("config.yaml")
X, y, X_test, test_ids = load_data(config)

In [32]:
# Хелпер: собрать модель из конфига, прогнать CV и записать результат в журнал
def run_model(cfg, name=None):
    model = build_model(cfg)
    scores, _ = cross_validate(model, X, y, X_test, cfg)
    features = list(cfg.features.numeric) + list(cfg.features.categorical) + list(cfg.features.log_numeric)
    params = OmegaConf.to_container(cfg.model.params, resolve=True)
    log_result(cfg, name or cfg.model.name, params, features, scores)
    return float(round(scores.mean(), 4)), float(round(scores.std(), 4))

## Бейзлайн

In [7]:
# 3.2 Бейзлайн: всем домам медианная цена
from sklearn.dummy import DummyRegressor

scores, _ = cross_validate(DummyRegressor(strategy="median"), X, y, X_test, config)
log_result(config, "median", {}, [], scores)
round(scores.mean(), 4), round(scores.std(), 4)

(np.float64(0.3998), np.float64(0.0118))

In [8]:
# 3.3 Линейная регрессия на 4 сильнейших признаках: журнал и сабмит
print(run_model(config))
_, test_pred = cross_validate(build_model(config), X, y, X_test, config)
save_submission(test_ids, test_pred, config, "baseline_linreg_4features")

(np.float64(0.1842), np.float64(0.0296))


'outputs\\submission_baseline_linreg_4features.csv'

## Предобработка

In [7]:
# 4.1 Все 79 признаков как есть, Ridge(alpha=10): точка отсчёта
print(run_model(config))

(np.float64(0.1445), np.float64(0.0425))


In [12]:
# 4.2 Проверка: какие пропуски остались
print(X.isna().sum()[X.isna().sum() > 0])
print(X_test.isna().sum()[X_test.isna().sum() > 0])

LotFrontage    259
Electrical       1
GarageYrBlt     81
dtype: int64
MSZoning         4
LotFrontage    227
Utilities        2
Exterior1st      1
Exterior2nd      1
KitchenQual      1
Functional       2
GarageYrBlt     78
SaleType         1
dtype: int64


In [13]:
# 4.2 NaN «объекта нет» → "None" / 0
print(run_model(config, "ridge_fill_absent"))

(np.float64(0.1447), np.float64(0.0428))


In [25]:
# 4.3 Проверка: порядковые столбцы стали числами
print(X[["KitchenQual", "FireplaceQu", "BsmtExposure", "Functional"]].head())
print(X["KitchenQual"].value_counts())

   KitchenQual  FireplaceQu  BsmtExposure  Functional
0            4            0             1           7
1            3            3             4           7
2            4            3             2           7
3            4            4             1           7
4            4            3             3           7
KitchenQual
3    735
4    586
5    100
2     39
Name: count, dtype: int64


In [26]:
# 4.3 Порядковые категории → числа
print(run_model(config, "ridge_ordinal"))

(np.float64(0.143), np.float64(0.0419))


In [29]:
# 4.4 MSSubClass → категория
print(run_model(config, "ridge_mssubclass_cat"))

(np.float64(0.1421), np.float64(0.0421))


### Выводы 4.1–4.4

| Шаг | Изменение | CV RMSE |
| --- | --- | --- |
| 4.1 | все 79 признаков как есть, Ridge(alpha=10) | 0.1445 ± 0.0425 |
| 4.2 | NaN «объекта нет» → "None" / 0 | 0.1447 ± 0.0428 |
| 4.3 | порядковые категории → числа (без PoolQC) | 0.1430 ± 0.0419 |
| 4.4 | MSSubClass → категория | 0.1421 ± 0.0421 |

- 4.2 по CV ничего не дал: информация «объекта нет» уже была в других столбцах (Fireplaces = 0, GarageCars = 0).
- 4.3: PoolQC убрали. Бассейн есть у 7 домов, после StandardScaler значение уходило на ~17 стандартных отклонений, а выброс 1299 учил модель «бассейн = дешевле». С PoolQC CV был 0.1584.
- 4.4: заполнение LotFrontage медианой по району проверили, эффекта нет (0.1421 → 0.1421). Оставили общую медиану.
- Разброс ± 0.04 почти целиком даёт один фолд с выбросами 524 и 1299, поэтому эффекты небольших изменений пока плохо видны.

In [33]:
# 4.5 Скошенные площади → log_numeric
print(run_model(config, "ridge_log_areas"))

(0.1332, 0.0328)


In [35]:
# 4.6 Без выбросов 524 и 1299
print(X.shape, y.shape)
print(run_model(config, "ridge_drop_outliers"))

(1458, 79) (1458,)
(0.1116, 0.0119)


In [38]:
# 4.7 Без почти константных, дублей и почти нулевых признаков + сабмит
print(run_model(config, "ridge_step4_final"))
_, test_pred = cross_validate(build_model(config), X, y, X_test, config)
save_submission(test_ids, test_pred, config, "ridge_preprocessed")

(0.1112, 0.0114)


'outputs\\submission_ridge_preprocessed.csv'

### Выводы 4.5–4.7 и итог предобработки

| Шаг | Изменение | CV RMSE |
| --- | --- | --- |
| 4.5 | 9 скошенных площадей → log1p | 0.1332 ± 0.0328 |
| 4.6 | убрали выбросы 524 и 1299 из train | 0.1116 ± 0.0119 |
| 4.7 | убрали 13 почти константных, дублирующих и почти нулевых признаков | 0.1112 ± 0.0114 |

- 4.5 дал самый большой прирост: цена в логарифме, и площади в логарифме дают связь «процент к проценту». Заодно сжался хвост огромных площадей.
- 4.6: разброс по фолдам упал втрое. Часть прироста CV «бесплатная»: трудные дома ушли из валидации. В test остался похожий дом (Id 2550), поэтому Kaggle хуже CV.
- 4.7: качество то же, признаков 65 вместо 78. При равном качестве выбираем более простую модель.

**Итог шага 4:** Ridge(alpha=10) CV 0.1445 → 0.1112, Kaggle 0.18030 (бейзлайн на 4 признаках) → 0.12569. CV стал примерно на 0.015 оптимистичнее Kaggle.